In [0]:
-- TRADES pipeline v4 — add cluster fields via league name where helpful.
CREATE OR REFRESH MATERIALIZED VIEW stg_trade_transactions AS
SELECT
  t.league_id,
  li.season,
  t.transaction_id,
  t.status,
  t.creator,
  t.adds,
  t.drops,
  t.roster_ids,
  t.draft_picks,
  t.leg AS week,
  t.created,
  to_timestamp(t.created/1000.0) AS event_ts,
  lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
  li.name AS cluster_name
FROM workspace.sleeper_raw.sleeper_transactions_snapshot t
JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
WHERE t.type = 'trade';

CREATE OR REFRESH MATERIALIZED VIEW bridge_trade_transaction_season AS
SELECT league_id, transaction_id, season, week, event_ts, cluster_key, cluster_name
FROM stg_trade_transactions;

CREATE OR REFRESH MATERIALIZED VIEW dim_season_week_index AS
SELECT DISTINCT season, week FROM workspace.sleeper_core.fact_team_week;

-- Player assets (incoming/outgoing)
CREATE OR REFRESH MATERIALIZED VIEW fact_trade_assets_player_only AS
WITH tx AS (SELECT * FROM stg_trade_transactions),
adds AS (
  SELECT league_id, season, transaction_id, week, event_ts, cluster_key, cluster_name,
         explode(transform(map_entries(adds), e -> named_struct('player_id', e.key, 'side_roster_id', e.value, 'direction', 'incoming'))) AS a
  FROM tx
),
drops AS (
  SELECT league_id, season, transaction_id, week, event_ts, cluster_key, cluster_name,
         explode(transform(map_entries(drops), e -> named_struct('player_id', e.key, 'side_roster_id', e.value, 'direction', 'outgoing'))) AS d
  FROM tx
)
SELECT league_id, season, transaction_id, week, event_ts, cluster_key, cluster_name, a.player_id, a.side_roster_id, a.direction FROM adds
UNION ALL
SELECT league_id, season, transaction_id, week, event_ts, cluster_key, cluster_name, d.player_id, d.side_roster_id, d.direction FROM drops;

CREATE OR REFRESH MATERIALIZED VIEW fact_trade_faab_by_side AS
SELECT league_id, season, transaction_id, week, event_ts, cluster_key, cluster_name,
       explode(roster_ids) AS side_roster_id,
       0 AS faab_delta
FROM stg_trade_transactions;

CREATE OR REFRESH MATERIALIZED VIEW fact_trade_post_points AS
SELECT a.league_id, a.transaction_id, a.side_roster_id, a.player_id,
       p.season, p.week, p.points AS realized_points,
       a.cluster_key, a.cluster_name
FROM fact_trade_assets_player_only a
JOIN workspace.sleeper_core.fact_player_week p
  ON  a.league_id = p.league_id
  AND a.season    = p.season
  AND a.side_roster_id = p.roster_id
  AND p.player_id = a.player_id
WHERE a.direction = 'incoming'
  AND p.week > a.week;

CREATE OR REFRESH MATERIALIZED VIEW fact_trade_post_points_horizon AS
WITH trade_side_week AS (
  SELECT DISTINCT league_id, transaction_id, side_roster_id, week AS trade_week
  FROM fact_trade_assets_player_only
),
joined AS (
  SELECT b.*, (b.week - s.trade_week) AS weeks_after
  FROM fact_trade_post_points b
  JOIN trade_side_week s
    ON  b.league_id = s.league_id
    AND b.transaction_id = s.transaction_id
    AND b.side_roster_id = s.side_roster_id
)
SELECT * FROM joined WHERE weeks_after BETWEEN 1 AND 4;

-- Draft pick mapping
CREATE OR REFRESH MATERIALIZED VIEW bridge_trade_pick_mapping_complete AS
WITH tx AS (
  SELECT league_id, transaction_id, draft_picks FROM stg_trade_transactions
),
parsed AS (
  SELECT league_id, transaction_id,
         transform(cast(draft_picks AS array<string>), p -> from_json(p, 'MAP<STRING,STRING>')) AS picks
  FROM tx
),
rows AS (
  SELECT league_id, transaction_id, explode(picks) AS mp FROM parsed
),
norm AS (
  SELECT league_id, transaction_id,
         CAST(mp['round'] AS INT)       AS round,
         CAST(mp['pick_no'] AS INT)     AS overall_pick,
         CAST(mp['owner_id'] AS INT)    AS from_roster_id,
         CAST(mp['roster_id'] AS INT)   AS to_roster_id,
         mp['season']                   AS realized_season
  FROM rows
)
SELECT n.league_id, n.transaction_id, n.round, n.overall_pick, n.from_roster_id, n.to_roster_id,
       COALESCE(n.realized_season, CAST(CAST(li.season AS INT)+1 AS STRING)) AS realized_season
FROM norm n
JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id);

-- Pick realization WITHOUT self-join
CREATE OR REFRESH MATERIALIZED VIEW fact_trade_pick_realization AS
WITH mapping AS (
  SELECT
    b.league_id,
    b.transaction_id,
    b.to_roster_id      AS side_roster_id,
    b.realized_season   AS season,
    b.round,
    b.overall_pick
  FROM bridge_trade_pick_mapping_complete b
),
draft_lookup AS (
  SELECT
    dp.league_id,
    dp.pick_no,
    dp.round,
    dr.season,
    dp.player_id
  FROM workspace.sleeper_raw.sleeper_draft_picks_snapshot dp
  JOIN workspace.sleeper_raw.sleeper_drafts_snapshot dr
    ON dp.draft_id = dr.draft_id
),
joined AS (
  SELECT
    m.league_id,
    m.transaction_id,
    m.side_roster_id,
    m.season,
    dl.player_id,
    COALESCE(SUM(fp.points), 0.0) AS realized_points
  FROM mapping m
  LEFT JOIN draft_lookup dl
    ON  m.league_id  = dl.league_id
    AND m.round      = dl.round
    AND m.overall_pick = dl.pick_no
    AND m.season     = dl.season
  LEFT JOIN workspace.sleeper_core.fact_player_week fp
    ON  fp.league_id = m.league_id
    AND fp.season    = m.season
    AND fp.roster_id = m.side_roster_id
    AND fp.player_id = dl.player_id
  GROUP BY m.league_id, m.transaction_id, m.side_roster_id, m.season, dl.player_id
)
SELECT
  league_id,
  transaction_id,
  side_roster_id,
  season,
  player_id,
  realized_points
FROM joined;

CREATE OR REFRESH MATERIALIZED VIEW agg_trade_impact_per_season AS
WITH post AS (
  SELECT league_id, transaction_id, side_roster_id, season, cluster_key, cluster_name,
         SUM(realized_points) AS sum_post_points
  FROM fact_trade_post_points_horizon
  GROUP BY league_id, transaction_id, side_roster_id, season, cluster_key, cluster_name
),
faab AS (
  SELECT league_id, transaction_id, side_roster_id, season, cluster_key, cluster_name,
         SUM(faab_delta) AS sum_faab_delta
  FROM fact_trade_faab_by_side
  GROUP BY league_id, transaction_id, side_roster_id, season, cluster_key, cluster_name
),
pick AS (
  SELECT league_id, transaction_id, side_roster_id, season,
         SUM(realized_points) AS sum_pick_points
  FROM fact_trade_pick_realization
  GROUP BY league_id, transaction_id, side_roster_id, season
)
SELECT
  coalesce(post.league_id, faab.league_id, pick.league_id) AS league_id,
  coalesce(post.transaction_id, faab.transaction_id, pick.transaction_id) AS transaction_id,
  coalesce(post.side_roster_id, faab.side_roster_id, pick.side_roster_id) AS side_roster_id,
  coalesce(post.season, faab.season, pick.season) AS season,
  coalesce(post.cluster_key, faab.cluster_key) AS cluster_key,
  coalesce(post.cluster_name, faab.cluster_name) AS cluster_name,
  coalesce(post.sum_post_points, 0.0) AS sum_post_points,
  coalesce(pick.sum_pick_points, 0.0) AS sum_pick_points,
  coalesce(faab.sum_faab_delta, 0) AS sum_faab_delta,
  coalesce(post.sum_post_points, 0.0) + coalesce(pick.sum_pick_points, 0.0) AS season_impact_points,
  coalesce(post.sum_post_points, 0.0) + coalesce(pick.sum_pick_points, 0.0) + coalesce(faab.sum_faab_delta, 0) AS season_impact_total
FROM post
FULL OUTER JOIN faab
  ON  post.league_id = faab.league_id
  AND post.transaction_id = faab.transaction_id
  AND post.side_roster_id = faab.side_roster_id
  AND post.season = faab.season
FULL OUTER JOIN pick
  ON  coalesce(post.league_id, faab.league_id) = pick.league_id
  AND coalesce(post.transaction_id, faab.transaction_id) = pick.transaction_id
  AND coalesce(post.side_roster_id, faab.side_roster_id) = pick.side_roster_id
  AND coalesce(post.season, faab.season) = pick.season;

CREATE OR REFRESH MATERIALIZED VIEW agg_trade_impact_all_time AS
SELECT league_id, side_roster_id,
       SUM(season_impact_points) AS total_points,
       SUM(sum_faab_delta)       AS total_faab,
       SUM(season_impact_total)  AS total_impact
FROM agg_trade_impact_per_season
GROUP BY league_id, side_roster_id;
